# 📑 Hoist Scheduled Payments Report
 
## 🔗 Links
- **[BRD Scheduled Payments - TBD](tbd)**  
  Comprehensive documentation outlining the business requirements for this report.
  
- **[Story Ticket - TBD](tbd)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket - TBD]()**  
  A summary of findings from UAT.
 
---

In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter

env = get_env_prefix()

In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")

if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    now_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    now_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
    run_date = spark.sql(f"SELECT CurrentMonthLastWorkingDate FROM `{env}-cdh`.reference.ref_calendar WHERE CalendarDate = last_day(add_months(to_date('{now_date}'), -1))").first()["CurrentMonthLastWorkingDate"]

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope - All HOIST portfolios
PortfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    

DecomissionDate = "2026-06-04"

FirstOfCurrentMonth = spark.sql(f"""
    SELECT CAST(DATE_TRUNC('MONTH', '{run_date}') AS DATE)
""").collect()[0][0]

RefDateAfter2Y = spark.sql(f"""
    SELECT CAST(DATEADD(DATE_TRUNC('MONTH', DATEADD(MONTH,15,'{run_date}')), -1) AS DATE)
""").collect()[0][0]

##### Views

In [0]:
migrationDate = spark.sql(f"""
          SELECT case when PortfolioName = 'Mercury' then '2019-07-01'
                else min(cast(AssignmentDate as date)) end AS migrationDate,
          PortfolioName
          FROM `{env}-cdh`.primary.portfolio
          WHERE PortfolioName in {PortfolioName}
          GROUP BY PortfolioName
      """)
migrationDate.createOrReplaceTempView("migrationDate")

reassignments = spark.sql(f"""
        select distinct re.LoanID
        from `{env}-reportinputfiles`.investor.hoist_reassignments as re
          """)
reassignments.createOrReplaceTempView("reassignments")

dexposure = spark.sql(f"""
         SELECT *,
         '{run_date}' AS RunDate
         FROM `{env}-cdh`.derivative.get_all_exposures_by_business_date('{run_date}')
         WHERE sourcesystem = 'profits'
         AND PortfolioName in {PortfolioName}
         """)
dexposure.createOrReplaceTempView("dexposure")                  

dpaymentschedule = spark.sql(f"""      
        select * from(
            select ps.* ,  rank() over (partition by ps.settlementkey order by ps.BusinessDate desc) as Rank
            from `{env}-cdh`.primary.paymentschedule as ps
            where ps.BusinessDate <= '{run_date}'
            AND ps.PortfolioName in {PortfolioName}
            and ps.PaymentScheduleType = 'Settlement Payment Schedule'
        ) where rank = 1
          """)
dpaymentschedule.createOrReplaceTempView("dpaymentschedule")

settlementexposure = spark.sql(f"""         
        select * from (
            select *,  rank() over (partition by EXPOSUREKEY, CDI order by BusinessDate desc) as Rank
            from `p-cdh`.derivative.dsettlementexposure         -- set to `p-cdh` as table does not exist in production
            WHERE BusinessDate <= '{run_date}'                                    
            AND PortfolioName in {PortfolioName}
            AND SettlementType not in ('Άρση Εγγυητικής Σχέσης', 'ΑΡΣΗ ΕΓΓΥΗΣΗΣ')
            and exposurekey is not null
        ) where rank = 1 AND CAST(ImplementationDate AS DATE) <= '{run_date}'
  """)
settlementexposure.createOrReplaceTempView("settlementexposure")

dunderlaw = spark.sql(f"""
        SELECT CDI, CASE WHEN InLawFlag = 1 THEN 1 ELSE 0 END AS L3869Flag
        FROM (
            SELECT 
                    CDI,
                    InLawFlag,
                   UnderLawType,
                    ROW_NUMBER() OVER (
                            PARTITION BY CDI
                            ORDER BY BusinessDate DESC
                    ) AS rn
            FROM `{env}-cdh`.derivative.dunderlaw
            WHERE BusinessDate <=  '{run_date}'      
            AND PortfolioName in {PortfolioName}
            AND UnderLawType LIKE '%3869%'
        ) t
        WHERE rn = 1
  """)        
dunderlaw.createOrReplaceTempView("dunderlaw")

##### Aggregations

In [0]:
dpaymentschedule_Agg = spark.sql(f"""
        select PS.SettlementKey,
            SUM(CASE WHEN CAST(PS.InstallmentDate AS DATE) <= '{run_date}' THEN InstallmentAmount ELSE 0 END) AS ScheduledAmount,
            SUM(CASE WHEN CAST(PS.InstallmentDate AS DATE) < '{FirstOfCurrentMonth}' AND CAST(PS.InstallmentDate AS DATE) > md.migrationDate THEN InstallmentAmount ELSE 0 END) AS ScheduleBeforeReference,
            SUM(CASE WHEN CAST(PS.InstallmentDate AS DATE) > '{RefDateAfter2Y}' AND CAST(PS.InstallmentDate AS DATE) > md.migrationDate THEN InstallmentAmount ELSE 0 END) AS ScheduleAfter2Y
        from dpaymentschedule AS PS
        LEFT JOIN migrationDate AS MD ON PS.PortfolioName = MD.PortfolioName
        GROUP BY PS.SettlementKey
        """)
dpaymentschedule_Agg.createOrReplaceTempView("dpaymentschedule_Agg")

dpaymentschedule_Monthly = spark.sql(f"""
        select 
            PS.SettlementKey,   
            MONTH(ps.InstallmentDate) as InstallmentMonth, 
            YEAR(ps.InstallmentDate) as InstallmentYear,
            SUM(InstallmentAmount ) AS InstallmentAmount
        from dpaymentschedule AS PS
        LEFT JOIN migrationDate AS MD ON PS.PortfolioName = MD.PortfolioName
        where CAST(PS.InstallmentDate AS DATE) > md.migrationDate
        GROUP BY PS.SettlementKey, MONTH(ps.InstallmentDate), YEAR(ps.InstallmentDate)
        """)
dpaymentschedule_Monthly.createOrReplaceTempView("dpaymentschedule_Monthly")

dpaymentscheduleperimeter = spark.sql(f"""
        select distinct PS.SettlementKey 
        from dpaymentschedule as ps
        left join migrationDate as md on md.portfolioname = ps.portfolioname
        where ps.InstallmentDate >= md.migrationDate
          """)
dpaymentscheduleperimeter.createOrReplaceTempView("dpaymentscheduleperimeter")


In [0]:
Final = spark.sql(f"""
        select distinct
            CAST('{run_date}' AS DATE) as BusinessDate,
            dexp.PortfolioName,
            dexp.ExposureKey,
            dexp.AgreementKey,
            dexp.PartyTaxRegistrationNumber ,
            dexp.CDI ,
            dexp.AssetSegment,
            ifnull(d.L3869Flag, 0) as L3869Flag,
            CAST(dexp.InitialAssignmentAmount AS DECIMAL(18,2)) as TotalFundedBalance,
            CAST(PSA.ScheduledAmount AS DECIMAL(18,2)) as TotalAmountDue, 
            CAST(se.ProposedAmount AS DECIMAL(18,2)) AS ApplicationAmount, 
            CAST(PSA.ScheduleBeforeReference AS DECIMAL(18,2)) as InstallmentAmountBeforeReferenceDate, 
            CAST(PSA.ScheduleAfter2Y AS DECIMAL(18,2)) as InstallmentAmountAfter2Years, 
            CAST(IFNULL(PSM.InstallmentAmount,0) AS DECIMAL(18,2)) AS InstallmentAmount, 
            CASE WHEN PSM.InstallmentYear IS NOT NULL THEN LAST_DAY(CAST(CONCAT(PSM.InstallmentYear,'-',PSM.InstallmentMonth,'-01') AS DATE)) 
                ELSE '9999-12-31' END	AS InstallmentDate
                , se.settlementstatus                                     
                ,se.IsMigratedFlag                                 
        from dexposure AS dexp
        INNER JOIN settlementexposure AS SE ON SE.ExposureKey = dexp.ExposureKey
        INNER JOIN dpaymentschedulePerimeter AS PSP ON PSP.SettlementKey = SE.SettlementKey
        LEFT JOIN dpaymentschedule_Agg AS PSA ON PSA.SettlementKey = SE.SettlementKey
        LEFT JOIN dpaymentschedule_Monthly AS PSM ON PSM.SettlementKey = SE.SettlementKey
        LEFT JOIN dunderlaw d on dexp.CDI = d.CDI 
        LEFT JOIN reassignments as re on re.LoanID = dexp.ExposureKey
        WHERE RE.LoanID IS NULL AND (dexp.JUSTIFICATIONCODE IS NULL OR dexp.JUSTIFICATIONCODE NOT IN (43055,43056))
        AND dexp.ExposureKey not in ('MRC4532120065329000' ,'ORB36711908082001' ,'ORB989008024697505' ,'ORB989008131189506' ,'ORB989008331415408' ,'ORB989008340168200' ,'ORB989008369635203' ,'ORB989008506824108' ,'ORB4509031671737008' ,'ORB4509034509335003' ,'ORB5100990646896001' ,'ORB5100999062155008' ,'ORB5100999067178005' ,'ORB9991850850078692') -- cases with installment dates far in the future
        """)
Final.createOrReplaceTempView("Final")

##### Append Table

In [0]:
target_table_path_ScheduledPayments = f"`{env}-cdh-delivery`.investorreports.hoist_scheduledpayments"

spark.sql(f"""DELETE FROM {target_table_path_ScheduledPayments} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

Final = Final.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Final.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_ScheduledPayments) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_ScheduledPayments} SYNC METADATA")